In [23]:
#Getting environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
import os
import google.generativeai as genai
# Set up Gemini API client
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.1-flash-lite')


In [25]:
# Helper Function 1: Add a user message 
def add_user_message(history, user_input):
    history.append({
        'role': 'user',
        'parts': [user_input]  
    })

# Helper Function 2: Add an assistant message
def add_assistant_message(history, assistant_response):
    history.append({
        'role': 'assistant',
        'parts': [assistant_response]  
    })

# Helper Function 3: To get message from the model
def chat(history,stop_sequences=None,max_output_tokens=2048,temperature=0.7):
    generation_options = {
        'stop_sequences': stop_sequences,
        'max_output_tokens': max_output_tokens,
        'temperature': temperature
    }
    response = model.generate_content(
        history,
        generation_config=generation_options,
        stream=True
    )

    full_response = ""
    print("Assistant: ", end="", flush=True)
    for chunk in response:
        full_response += chunk.text
        print(chunk.text, end="", flush=True)
    print() # Adds a clean closing line breaks after streaming finishes
    return full_response
    

In [26]:
# Creating a dataset generation function
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [27]:
# Testing the dataset generation function
import json
dataset = generate_dataset()
print(dataset)

Assistant: 
[
  {
    "task": "Write a Python function that uses the Boto3 library to list all S3 bucket names in the current AWS account."
  },
  {
    "task": "Write a JSON IAM policy document that grants read-only access to a specific S3 bucket named 'my-app-data-bucket'."
  },
  {
    "task": "Write a Regex pattern that validates an AWS resource ARN for an EC2 instance, ensuring it follows the format 'arn:aws:ec2:region:account-id:instance/instance-id'."
  }
]

[{'task': 'Write a Python function that uses the Boto3 library to list all S3 bucket names in the current AWS account.'}, {'task': "Write a JSON IAM policy document that grants read-only access to a specific S3 bucket named 'my-app-data-bucket'."}, {'task': "Write a Regex pattern that validates an AWS resource ARN for an EC2 instance, ensuring it follows the format 'arn:aws:ec2:region:account-id:instance/instance-id'."}]


In [28]:
# Saving the dataset to a JSON file
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [29]:
# Added a run prompt function to take teh test case and merge with prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [30]:
# A test case function to run the prompt on a single test case and give it the score 
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [31]:
# A run eval function to run the test case function on all test cases in the dataset and return results
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [32]:
# Running the evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Assistant: To list all S3 buckets in your AWS account using Boto3, you need to use the `list_buckets` method provided by the S3 client.

### Prerequisites
1. **Install Boto3:** `pip install boto3`
2. **Configure AWS Credentials:** Ensure your environment is configured with credentials (via `aws configure`, environment variables, or IAM roles).

### The Python Code

```python
import boto3

def list_s3_buckets():
    """
    Returns a list of all S3 bucket names in the current AWS account.
    """
    # Initialize the S3 client
    s3_client = boto3.client('s3')
    
    try:
        # Call the API to list buckets
        response = s3_client.list_buckets()
        
        # Extract the bucket names from the response
        buckets = [bucket['Name'] for bucket in response['Buckets']]
        return buckets
    
    except Exception as e:
        print(f"Error listing buckets: {e}")
        return []

# Example usage:
if __name__ == "__main__":
    bucket_names = list_s3_buckets()
    p

In [33]:
print(json.dumps(results, indent=2))

[
  {
    "output": "To list all S3 buckets in your AWS account using Boto3, you need to use the `list_buckets` method provided by the S3 client.\n\n### Prerequisites\n1. **Install Boto3:** `pip install boto3`\n2. **Configure AWS Credentials:** Ensure your environment is configured with credentials (via `aws configure`, environment variables, or IAM roles).\n\n### The Python Code\n\n```python\nimport boto3\n\ndef list_s3_buckets():\n    \"\"\"\n    Returns a list of all S3 bucket names in the current AWS account.\n    \"\"\"\n    # Initialize the S3 client\n    s3_client = boto3.client('s3')\n    \n    try:\n        # Call the API to list buckets\n        response = s3_client.list_buckets()\n        \n        # Extract the bucket names from the response\n        buckets = [bucket['Name'] for bucket in response['Buckets']]\n        return buckets\n    \n    except Exception as e:\n        print(f\"Error listing buckets: {e}\")\n        return []\n\n# Example usage:\nif __name__ == \"__m